```java

    private ActivityMainBinding binding;
    private TextView flagText;
    private TextView inputText;
    private Button submitButton;
    private String userText;
    private String logTag = "NONESHALLPASS";
    private byte[] encIv = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15};
    private String encryptedFlag = "SowTrUiy0uSeQrZxZOvp5mYd4yAvPB+UhVwFzHU/ULetHhO2mlulk0Zjx9l+FaNhQMgCbWoM/m/BzOc+ZfHppnVkZ4s5oTT3d9KtFYdb6b0=";

    public native boolean checkPassword(String str);

    static {
        System.loadLibrary("noneshallpass");
    }

    private String decryptFlag(String str) throws NoSuchPaddingException, NoSuchAlgorithmException, InvalidKeyException, InvalidAlgorithmParameterException {
        try {
            SecretKeySpec secretKeySpec = new SecretKeySpec(str.getBytes(StandardCharsets.UTF_8), "AES");
            IvParameterSpec ivParameterSpec = new IvParameterSpec(this.encIv);
            Cipher cipher = Cipher.getInstance("AES/CBC/PKCS7Padding");
            cipher.init(2, secretKeySpec, ivParameterSpec);
            return new String(cipher.doFinal(Base64.decode(this.encryptedFlag.getBytes(StandardCharsets.UTF_8), 0)), "UTF-8");
        } catch (UnsupportedEncodingException | InvalidAlgorithmParameterException | InvalidKeyException | NoSuchAlgorithmException | BadPaddingException | IllegalBlockSizeException | NoSuchPaddingException e) {
            Log.w(this.logTag, "Failed to decrypt flag with " + e);
            return "";
        }
    }

    @Override // androidx.fragment.app.FragmentActivity, androidx.activity.ComponentActivity, androidx.core.app.ComponentActivity, android.app.Activity
    protected void onCreate(Bundle bundle) {
        super.onCreate(bundle);
        ActivityMainBinding activityMainBindingInflate = ActivityMainBinding.inflate(getLayoutInflater());
        this.binding = activityMainBindingInflate;
        setContentView(activityMainBindingInflate.getRoot());
        Button button = (Button) findViewById(R.id.submitButton);
        this.submitButton = button;
        button.setOnClickListener(new View.OnClickListener() { // from class: io.brewfault.noneshallpass.MainActivity$$ExternalSyntheticLambda0
            /* JADX DEBUG: Don't trust debug lines info. Lines numbers was adjusted: min line is 0 */
            @Override // android.view.View.OnClickListener
            public final void onClick(View view) throws NoSuchPaddingException, NoSuchAlgorithmException, InvalidKeyException, InvalidAlgorithmParameterException {
                this.f$0.m146lambda$onCreate$0$iobrewfaultnoneshallpassMainActivity(view);
            }
        });
    }

    /* renamed from: lambda$onCreate$0$io-brewfault-noneshallpass-MainActivity, reason: not valid java name */
    /* synthetic */ void m146lambda$onCreate$0$iobrewfaultnoneshallpassMainActivity(View view) throws NoSuchPaddingException, NoSuchAlgorithmException, InvalidKeyException, InvalidAlgorithmParameterException {
        TextView textView = (TextView) findViewById(R.id.submitText);
        this.inputText = textView;
        this.userText = textView.getText().toString();
        this.flagText = (TextView) findViewById(R.id.flagTextBox);
        if (checkPassword(this.userText)) {
            this.flagText.setText("Password correct! The flag is:\n" + decryptFlag(this.userText));
        } else {
            this.flagText.setText("I MOVE FOR NO MAN");
        }
    }
```

So, `noneshallpass` is a lib that executes `checkPassword` where the password is the key to decrypt the flag in `AES/CBC/PKCS7Padding`

We can decompile that app with jadx and retrieve the lib. The lib is apparently written in c++. This is the entry

```c
  uint64_t Java_io_brewfault_noneshallpass_MainActivity_checkPassword(int64_t* arg1)

      uint64_t x21 = _ReadMSR(SystemReg: tpidr_el0)
      int64_t x8 = *(x21 + 0x28)
      char* x0 = (*(*arg1 + 0x548))()
      uint8_t* x0_2 = malloc(bytes: strlen(x0))
      // 
      doEncryption(*montyPythonKey, x0, x0_2) // key:0x416a97, do decrpytion looks so wild, like some sub cipher
      simple_b64_encode(x0_2, strlen(x0)) 
      int64_t i = 0
      int32_t x22 = 0
      char var_60
      
      do
          uint64_t x20_1 = *(passwords + i)
          size_t x0_8 = strlen(x20_1)
          uint64_t x8_3 = zx.q(var_60)
          uint64_t x8_4
          uint64_t var_58
          
          if ((x8_3.d & 1) == 0)
              x8_4 = x8_3 u>> 1
          else
              x8_4 = var_58
          
          if (x0_8 == x8_4 && std::__ndk1::basic_strin...d::__ndk1::allocator<char> >::compare(
                  &var_60, 0, -ffffffffffffffff, x20_1) == 0)
              x22 += 1
          
          i += 8
      while (i != 0x800)
      
      free(mem: x0_2)
      void* var_50
      
      if ((zx.d(var_60) & 1) != 0)
          operator delete(var_50)
      
      if (*(x21 + 0x28) == x8)
          return zx.q(x22 s> 0 ? 1 : 0)
      
      __stack_chk_fail()
      noreturn
```

The doEncryption is likely RC4: 

```c
  int64_t RC4Encryption(char const* key, char* input, uint8_t* output)

  {
      uint64_t x22 = _ReadMSR(tpidr_el0);
      uint8_t* output_1 = output;
      int64_t x8 = *(uint64_t*)(x22 + 0x28);
      char* input_1 = input;
      int32_t x0 = strlen(key);
      int64_t i = 0;
      uint64_t x9 = 0;
      int128_t sbox;
      __builtin_memcpy(&sbox, 
          "\x00\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0a\x0b\x0c\x0d\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f", 
          32);
      int128_t nousage?2;
      __builtin_strncpy(&nousage?2, 
          " !\"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~\x7f", 
          96);
      int128_t nousage?;
      __builtin_memcpy(&nousage?, 
          "\x80\x81\x82\x83\x84\x85\x86\x87\x88\x89\x8a\x8b\x8c\x8d\x8e\x8f\x90\x91\x92\x93\x94\x95\x96\x97\x98\x99\x9a\x9b\x9c\x9d\x9e\x9f\xa0\xa1\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xab\xac\xad\xae\xaf\xb0\xb1\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xbb\xbc\xbd\xbe\xbf\xc0\xc1\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xcb\xcc\xcd\xce\xcf\xd0\xd1\xd2\xd3\xd4\xd5\xd6\xd7\xd8\xd9\xda\xdb\xdc\xdd\xde\xdf\xe0\xe1\xe2\xe3\xe4\xe5\xe6\xe7\xe8\xe9\xea\xeb\xec\xed\xee\xef\xf0\xf1\xf2\xf3\xf4\xf5\xf6\xf7\xf8\xf9\xfa\xfb\xfc\xfd\xfe\xff", 
          128);
      
      do
      {
          uint32_t x12_1 = (uint32_t)*(uint8_t*)(&sbox + i);
          int32_t x9_2 = x9 + x12_1 + (uint32_t)key[(uint64_t)(i % x0)];
          int32_t x11_5;
          
          if (x9_2 < 0)
              x11_5 = x9_2 + 0xff;
          else
              x11_5 = x9_2;
          
          x9 = (uint64_t)(x9_2 - (x11_5 & 0xffffff00));
          *(uint8_t*)(&sbox + i) = *(uint8_t*)(&sbox + x9);
          i += 1;
          *(uint8_t*)(&sbox + x9) = (char)x12_1;
      } while (i != 0x100);
      
      size_t input_size = strlen(input_1);
      
      if (input_size)
      {
          int32_t x8_1 = 0;
          uint64_t x10_1 = 0;
          size_t i_1;
          
          do
          {
              int32_t x8_2;
              
              if (x8_1 + 1 < 0)
                  x8_2 = x8_1 + 0x100;
              else
                  x8_2 = x8_1 + 1;
              
              x8_1 = x8_1 + 1 - (x8_2 & 0xffffff00);
              int64_t x11_9 = (int64_t)x8_1;
              uint32_t x12_3 = (uint32_t)*(uint8_t*)(&sbox + x11_9);
              int32_t x10_2 = x10_1 + x12_3;
              int32_t x13_2;
              
              if (x10_2 < 0)
                  x13_2 = x10_2 + 0xff;
              else
                  x13_2 = x10_2;
              
              i_1 = input_size;
              input_size -= 1;
              x10_1 = (uint64_t)(x10_2 - (x13_2 & 0xffffff00));
              *(uint8_t*)(&sbox + x11_9) = *(uint8_t*)(&sbox + x10_1);
              *(uint8_t*)(&sbox + x10_1) = (char)x12_3;
              char x12_4 = *(uint8_t*)input_1;
              input_1 = &input_1[1];
              *(uint8_t*)output_1 =
                  x12_4 ^ *(uint8_t*)(&sbox + (uint64_t)(*(uint8_t*)(&sbox + x11_9) + x12_3));
              output_1 = &output_1[1];
          } while (i_1 != 1);
      }
      
      if (*(uint64_t*)(x22 + 0x28) == x8)
          return 0;
      
      __stack_chk_fail();
      /* no return */
  }
```

We can see this because of the SBox init:

```
In [7]: for i in range(256): print(chr(i) if chr(i) in string.printable else hex(i),end="")
0x00x10x20x30x40x50x60x70x80xe0xf0x100x110x120x130x140x150x160x170x180x190x1a0x1b0x1c0x1d0x1e0x1f !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~0x7f0x800x810x820x830x840x850x860x870x880x890x8a0x8b0x8c0x8d0x8e0x8f0x900x910x920x930x940x950x960x970x980x990x9a0x9b0x9c0x9d0x9e0x9f0xa00xa10xa20xa30xa40xa50xa60xa70xa80xa90xaa0xab0xac0xad0xae0xaf0xb00xb10xb20xb30xb40xb50xb60xb70xb80xb90xba0xbb0xbc0xbd0xbe0xbf0xc00xc10xc20xc30xc40xc50xc60xc70xc80xc90xca0xcb0xcc0xcd0xce0xcf0xd00xd10xd20xd30xd40xd50xd60xd70xd80xd90xda0xdb0xdc0xdd0xde0xdf0xe00xe10xe20xe30xe40xe50xe60xe70xe80xe90xea0xeb0xec0xed0xee0xef0xf00xf10xf20xf30xf40xf50xf60xf70xf80xf90xfa0xfb0xfc0xfd0xfe0xff
```

The key is `IHaveNoQuarrelWithYouGoodSir` as we just follow the pointer.

```c
char* montyPythonKey = 0x416a97 {"IHaveNoQuarrelWithYouGoodSir"}
```

So I actually don't know what the call `char* password = (*(uint64_t*)(*(uint64_t*)arg1 + 1352))();` does, but I just start to decrypt all password and check which one decrypts the flag. Out of the 256 passwords inside must be right.



In [137]:
import base64
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

# Fixed IV from Java code
_IV = bytes([0, 1, 2, 3, 4, 5, 6, 7,
              8, 9, 10, 11, 12, 13, 14, 15])

def decflag(key: bytes, encrypted_b64: str = "SowTrUiy0uSeQrZxZOvp5mYd4yAvPB+UhVwFzHU/ULetHhO2mlulk0Zjx9l+FaNhQMgCbWoM/m/BzOc+ZfHppnVkZ4s5oTT3d9KtFYdb6b0=") -> bytes:
    """
    Decrypts the Base64-encrypted flag using AES/CBC/PKCS7 and the given key.
    :param key: AES key bytes (must be 16/24/32 bytes long).
    :param encrypted_b64: Base64 string of encryptedFlag (from Java code).
    :return: Decrypted plaintext bytes.
    """
    ct = base64.b64decode(encrypted_b64)
    cipher = AES.new(key, AES.MODE_CBC, iv=_IV)
    pt = unpad(cipher.decrypt(ct), AES.block_size)
    return pt

In [138]:
with open("passwords.txt","r") as f:
    passwords = eval(f.read())

In [139]:
for i,pw in enumerate(passwords):
    try:
        base64.b64decode(pw)
    except:
        print(hex(i),pw)
        break

In [140]:
passwords = [
    base64.b64decode(pw) for pw in passwords
]

In [141]:
from collections import Counter

Counter(map(len,passwords)),[128/8,192/8,256/8]

(Counter({10: 178, 11: 50, 12: 15, 13: 8, 32: 1, 16: 1, 17: 1, 15: 1, 14: 1}),
 [16.0, 24.0, 32.0])

In [142]:
# First stage, RC4
from Crypto.Cipher import ARC4 

def decrypt(data):
    key = b"IHaveNoQuarrelWithYouGoodSir"
    instance = ARC4.new(key)
    return instance.decrypt(data)

passwords = list(map(decrypt,passwords))
passwords

[b'1234567890',
 b'basketball',
 b'tinkerbell',
 b'hellokitty',
 b'christopher',
 b'0123456789',
 b'volleyball',
 b'chrisbrown',
 b'strawberry',
 b'qwertyuiop',
 b'harrypotter',
 b'sweetheart',
 b'12345678910',
 b'manchester',
 b'linkinpark',
 b'bestfriend',
 b'estrellita',
 b'california',
 b'friendster',
 b'bestfriends',
 b'daddysgirl',
 b'tokiohotel',
 b'undertaker',
 b'cheerleader',
 b'cinderella',
 b'jesuschrist',
 b'ilovejesus',
 b'princesita',
 b'jesucristo',
 b'0987654321',
 b'butterfly1',
 b'friendship',
 b'simpleplan',
 b'diosesamor',
 b'beautiful1',
 b'mississippi',
 b'mickeymouse',
 b'prettygirl',
 b'elizabeth1',
 b'9876543210',
 b'watermelon',
 b'juancarlos',
 b'teamomucho',
 b'0000000000',
 b'falloutboy',
 b'spongebob1',
 b'1111111111',
 b'realmadrid',
 b'ronaldinho',
 b'password123',
 b'tequieromucho',
 b'chocolate1',
 b'tweetybird',
 b'smallville',
 b'ilovemyself',
 b'ilovechris',
 b'gymnastics',
 b'sailormoon',
 b'dramaqueen',
 b'computadora',
 b'liverpoolfc',
 b'121231

In [143]:
from collections import Counter

c = Counter(map(len,passwords))
c.keys()

dict_keys([10, 11, 13, 12, 32, 16, 17, 15, 14])

In [144]:
for pw in passwords:
    if len(pw) in [16,24,32]:
        print(pw.decode())

IAMAR7HURKINGOFTHEBRITONSJOINME!
manchesterunited


In [145]:
for pw in passwords:
    try:
        flag = decflag(pw)
        print(flag)
    except: pass

b'FLAG{Av3rage_w1ngsp3ed_velocity_of_unl@den_3urpean_sw4llow_is_20.1mph}'
